# SeraTune TRIBE v2 Worker

This notebook runs the TRIBE v2 inference model as a FastAPI server,
exposed via ngrok so the SeraTune backend can call it.

**Prerequisites:**
1. Runtime → Change runtime type → **T4 GPU**
2. A HuggingFace account with the LLaMA 3.2-3B license accepted
3. A HuggingFace access token
4. An ngrok auth token (free at https://ngrok.com)

**After running all cells, copy the ngrok URL and set it in your backend `.env`:**
```
TRIBE_WORKER_URL=https://xxxx-xx-xx.ngrok-free.app
USE_MOCK_TRIBE=false
```

## 1. Install dependencies

Run cells 1a and 1b in order. **After each cell, click Runtime → Restart runtime before running the next cell.**

In [ ]:
# Cell 1a: Uninstall conflicting packages
# ⚠️ After this cell finishes, click Runtime → Restart runtime
!pip uninstall -y torch torchaudio torchvision numpy neuralset neuraltrain tribev2 click gtts
print("\n" + "="*60)
print("⚠️  RESTART RUNTIME NOW: Runtime → Restart runtime")
print("    Then run the next cell (1b)")
print("="*60)

In [ ]:
# Cell 1b: Install compatible versions
# ⚠️ After this cell finishes, click Runtime → Restart runtime
!pip install -q \
  "numpy==2.2.6" \
  "torch==2.6.0" \
  "torchaudio==2.6.0" \
  "torchvision==0.21.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  "neuralset==0.0.2" \
  "neuraltrain==0.0.2" \
  "x_transformers==1.27.20" \
  "moviepy>=2.2.1" \
  gtts julius Levenshtein transformers huggingface_hub soundfile librosa langdetect spacy

!pip install -q git+https://github.com/facebookresearch/tribev2.git

!pip install -q fastapi uvicorn pyngrok python-multipart

print("\n" + "="*60)
print("⚠️  RESTART RUNTIME NOW: Runtime → Restart runtime")
print("    Then skip to the HuggingFace auth cell")
print("="*60)

## 2. Authenticate with HuggingFace

Paste your HuggingFace access token below. You need to have accepted the
[LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B) first.

In [ ]:
from huggingface_hub import login

# Replace with your token, or it will prompt you interactively
HF_TOKEN = ""  # paste your token here, or leave empty to be prompted

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

## 3. Load TRIBE v2 model

This downloads the model weights (~8GB) and loads them onto the GPU.
First run takes a few minutes.

In [ ]:
from tribev2 import TribeModel
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading TRIBE v2 model...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded!")

## 4. Quick test — verify inference works

In [ ]:
# Generate a short test audio file
import numpy as np
import wave
import struct

sample_rate = 16000
duration = 5  # seconds
t = np.linspace(0, duration, sample_rate * duration, endpoint=False)
audio_data = (16000 * np.sin(2 * np.pi * 440 * t)).astype(np.int16)

with wave.open("/tmp/test_tone.wav", "w") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes(audio_data.tobytes())

# Run inference
df = model.get_events_dataframe(audio_path="/tmp/test_tone.wav")
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}")  # (T, 20484)
print(f"Time segments: {len(segments)}")
print(f"Vertices per segment: {preds.shape[1]}")
print("\nInference test passed!")

## 5. Start the TRIBE worker API server

In [ ]:
%%writefile /tmp/tribe_worker.py
"""TRIBE v2 inference worker — lightweight FastAPI server."""

import logging
import os
import tempfile
import time

import numpy as np
from fastapi import FastAPI, File, HTTPException, UploadFile
from tribev2 import TribeModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="TRIBE v2 Worker", version="0.1.0")

# Model is loaded once at startup
_model: TribeModel | None = None


def get_model() -> TribeModel:
    global _model
    if _model is None:
        logger.info("Loading TRIBE v2 model...")
        _model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
        logger.info("Model loaded.")
    return _model


@app.on_event("startup")
async def startup():
    get_model()


@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": _model is not None}


@app.post("/analyze")
async def analyze(audio: UploadFile = File(...)):
    """Run TRIBE v2 inference on an uploaded audio file.

    Returns the raw predictions matrix as a nested list.
    Shape: (n_time_segments, 20484)
    """
    model = get_model()

    tmp_path = None
    try:
        suffix = os.path.splitext(audio.filename or "audio.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp_path = tmp.name
            content = await audio.read()
            tmp.write(content)

        logger.info("Running inference on %s (%d bytes)...", audio.filename, len(content))
        start = time.time()

        df = model.get_events_dataframe(audio_path=tmp_path)
        preds, segments = model.predict(events=df)

        elapsed = time.time() - start
        logger.info(
            "Inference complete: %s → shape %s in %.1fs",
            audio.filename, preds.shape, elapsed,
        )

        return {
            "preds": preds.tolist(),
            "shape": list(preds.shape),
            "n_segments": preds.shape[0],
            "n_vertices": preds.shape[1],
            "inference_time_s": round(elapsed, 2),
        }
    except Exception as e:
        logger.exception("Inference failed")
        raise HTTPException(500, f"Inference failed: {e}")
    finally:
        if tmp_path:
            os.unlink(tmp_path)

## 6. Set up ngrok tunnel

Get a free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # paste your ngrok auth token here

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Set up ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start the FastAPI server in a background thread
# (importing the module we wrote above)
import importlib.util
spec = importlib.util.spec_from_file_location("tribe_worker", "/tmp/tribe_worker.py")
tribe_worker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tribe_worker)

# Pre-load the model into the worker module AFTER exec_module
# (exec_module re-runs module-level code which sets _model = None)
tribe_worker._model = model

PORT = 8001

def run_server():
    uvicorn.run(tribe_worker.app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Open ngrok tunnel
public_url = ngrok.connect(PORT, "http").public_url

print("="*60)
print(f"TRIBE v2 Worker is running!")
print(f"")
print(f"Public URL: {public_url}")
print(f"")
print(f"Add this to your backend .env:")
print(f"  TRIBE_WORKER_URL={public_url}")
print(f"  USE_MOCK_TRIBE=false")
print(f"")
print(f"Health check: {public_url}/health")
print("="*60)

## 7. Test the worker endpoint

In [ ]:
import httpx
import time

# Wait a moment for the server to start
time.sleep(2)

# Test health
resp = httpx.get(f"{public_url}/health")
print(f"Health: {resp.json()}")

# Test inference with the test tone
print("\nRunning inference test via HTTP...")
with open("/tmp/test_tone.wav", "rb") as f:
    resp = httpx.post(
        f"{public_url}/analyze",
        files={"audio": ("test_tone.wav", f, "audio/wav")},
        timeout=180.0,
    )

data = resp.json()
print(f"Shape: {data['shape']}")
print(f"Segments: {data['n_segments']}")
print(f"Vertices: {data['n_vertices']}")
print(f"Inference time: {data['inference_time_s']}s")
print("\nWorker is ready! Keep this notebook running.")

## Keep alive

Run this cell to keep the notebook alive. The worker will stay running
as long as this cell is executing. Press the stop button to shut down.

In [ ]:
import time

print(f"Worker running at: {public_url}")
print("Press the stop button (or interrupt kernel) to shut down.")
print()

try:
    while True:
        time.sleep(60)
        print(f"[{time.strftime('%H:%M:%S')}] Worker still running at {public_url}")
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()